<div class="alert alert-info">
<h1>Beyond the standard GLM</h1>
<h4>Environmental Statistics 4</h4>
    
Carsten F. Dormann
</div>

There are two thing to learn today:
-  Using other implementations of GLMs.
-  Using optimisation to define and run your own GLM-like analysis.

## 1 Exercise 1: Negative binomial regression
The response variable of interest is days absent, daysabs. The variable math is the standardized math score
for each student. The variable prog is a three-level nominal variable indicating the type of instructional
program in which the student is enrolled.


In [14]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import statsmodels.api as sm
import statsmodels.formula.api as smf
from matplotlib.colors import ListedColormap
from scipy.optimize import minimize
from scipy.optimize import curve_fit
from tabulate import tabulate
from scipy.stats import t, norm
import rpy2

import os
os.environ["R_HOME"] = r"C:\Users\Ikedichi_Azuh\anaconda3\envs\envstats\lib\R"

import warnings

### 1.1 Load the data and look at the summary of it

In [15]:
# pip install pyreadstat
absent = pd.read_spss("nb_data.sav")
print(absent.describe())

                id        math     daysabs        prog
count   314.000000  314.000000  314.000000  314.000000
mean   1575.910828   48.267516    5.955414    2.213376
std     502.314832   25.362391    7.036958    0.651133
min    1001.000000    1.000000    0.000000    1.000000
25%    1079.250000   28.000000    1.000000    2.000000
50%    1158.500000   48.000000    4.000000    2.000000
75%    2077.750000   70.000000    8.000000    3.000000
max    2157.000000   99.000000   35.000000    3.000000


In [17]:
# using URL
url='https://stats.idre.ucla.edu/stat/stata/dae/nb_data.dta'
absentdata = pd.read_stata(url)
print(absentdata.describe())

                id        math     daysabs        prog
count   314.000000  314.000000  314.000000  314.000000
mean   1575.910889   48.267517    5.955414    2.213376
std     502.314819   25.362392    7.036958    0.651133
min    1001.000000    1.000000    0.000000    1.000000
25%    1079.250000   28.000000    1.000000    2.000000
50%    1158.500000   48.000000    4.000000    2.000000
75%    2077.750000   70.000000    8.000000    3.000000
max    2157.000000   99.000000   35.000000    3.000000



Two columns need conversion to factors:


In [10]:
# What are they? convert them to factors




### 1.2 Plot the data
Plot absence times as a function of math score and program enrolled, ideally next to each other as a "double plot", yet each panel should be **square**!


### 1.3 Fit a Poisson regression model
Use both program and math and their interaction as predictors.

Any hint of over/under-dispersion?




### 1.4 Do the same with a negative binomial model

In [11]:
# fit negative binomial




### 1.5 Now do the same with `vglm` from VGAM **(SKIP 1.5)**
This is not yet implemented in python. So we can call R codes to see the results.


###  1.6 Plot fitted regression
Finally, plot the data again, indicating by three different colours the three program levels (1-3). Then compute the fitted 3 (plus errors!) lines from negative binomial and add them to the plot.

Ideally, you would restrict the range of the predicted regression to the range of the data *for that program*. 

Finally, it is customary to indicate non-significant lines (i.e. all of them) as dashed rather than solid lines.

## 2 Exercise 2: Optimising a likelihood-based model
Fit a non-linear function to data. For a change, we invent data representing a noisy sine-wave, so that we can directly evaluate whether our optimiser actually worked and estimated the values correctly.

In [ ]:
x = np.linspace(0, 4 * np.pi, num=100)
np.random.seed(123)
y = 1 + 0.5 * np.sin(0.8 * x) + np.random.normal(0, scale=0.1, size=100)


plt.figure(figsize=(8, 4))
plt.plot(x, y, '-')
plt.xlabel("x")
plt.ylabel("y")
plt.show()


### 2.1 Fitting a non-linear regression by hand
First formulate a function to optimise, including the assumption of (1) which distribution the data are from, and (2) the parameters to be fitted.

In [10]:
def sinefun(params):
    a, b, c, log_s = params
    s = np.exp(log_s)
    yhat = a + b * np.sin(c * x)
    return np.sum(norm.logpdf(y, loc=yhat, scale=s))


params_init = np.array([1, 1, 1, 1])
sinefun(params_init) # check that it works, in principle


-195.06029123082712

A few things are worth remembering:

  * as the standard deviation *must* be positive, we use the exponential transformation to ensure this;
  * this function yields the log-likelihood of observing the data *y*, given the parameters a to s; we want to *maximise* this value during optimisation;
  * `yhat` is the estimated value computed by our non-normal function.
 


1.  Now fit this function using `minimize()` from `scipy.optimize`
2.  Now also compute the Hessian, which we can use to compute standard errors for our estimates. 
3.  Finally, we can plot the fitted model into the data:


In [18]:
# Hint: since we want to maximise, we have to return -(log-likelihood) in our function because max (log-likelihood)= min -(log-likelihood)

# 1. create the function 





In [19]:
# 2. Optimize the function and extract the Optimized parameter values (Hint: use the method="BFGS" in the minimize())







In [20]:
# 3. calculate the standard errors for the parameters from the Hessian matrix







In [21]:
# 4. finally, we can plot the fitted model into the data:






### 2.2 Non-linear least squares
In this case, with a normally distributed error, `Python` has a simpler function at its disposal `curve_fit`. Fit the function using `curve_fit`; are the results similar to the optimisation approach in the previous subsection?

In [22]:
# With Curve_fit, the process above can be done easily:
# How can this be done?




